In [1]:
import pandas as pd
import numpy as np
from time import perf_counter
from datasets import load_dataset
from memory_profiler import memory_usage
from tqdm import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, precision_recall_fscore_support
from transformers import DistilBertTokenizer, DistilBertModel


In [2]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PRETRAINED_MODEL_NAME = 'distilbert-base-uncased-finetuned-sst-2-english'
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5 

print(f"Using device: {DEVICE}")

Using device: cuda


In [3]:
ds = load_dataset("dair-ai/emotion", "split")

train_df = ds['train'].to_pandas()
val_df = ds['validation'].to_pandas()
test_df = ds['test'].to_pandas()

train_df

,text,label
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,3
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,3
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,1
15998,i feel like this was such a rude comment and i...,3


In [4]:
class MultiClassClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels  # Labels should be integers: 0, 1, 2, ..., num_classes-1
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)  # Changed to scalar tensor of type long
        }

In [5]:
class DistilBertForMultiClassClassification(nn.Module):
    def __init__(self, num_classes):
        super(DistilBertForMultiClassClassification, self).__init__()
        self.distilbert = DistilBertModel.from_pretrained(PRETRAINED_MODEL_NAME)
        self.pre_classifier = nn.Linear(768, 768)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(768, num_classes)  # Output size is num_classes
    
    def forward(self, input_ids, attention_mask):
        outputs = self.distilbert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        hidden_state = outputs[0][:, 0]  # CLS token
        pooled_output = self.pre_classifier(hidden_state)
        pooled_output = nn.ReLU()(pooled_output)
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits  # Return raw logits, no sigmoid

In [6]:
def get_metrics(y_true, y_pred):

    acc = accuracy_score(y_true, y_pred)

    precisions, recalls, f1s, supports = precision_recall_fscore_support(y_true, y_pred)

    return acc, precisions, recalls, f1s

In [7]:
def train_model(model, train_dataloader, val_dataloader, optimizer, criterion, scheduler=None, epochs=EPOCHS):
    best_val_loss = float('inf')
    start_train = perf_counter()
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        train_preds = []
        train_true = []
        
        progress_bar = tqdm(train_dataloader, desc=f'Epoch {epoch + 1}/{epochs}', leave=False)
        for batch in progress_bar:
            optimizer.zero_grad()
            
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)  # Shape: (batch_size,)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)  # Shape: (batch_size, num_classes)
            loss = criterion(outputs, labels)  # CrossEntropyLoss expects logits and long labels
            train_loss += loss.item()
            
            # Accumulate predictions and true labels for metrics
            preds = torch.argmax(outputs, dim=1).cpu().numpy()  # Get class indices
            train_preds.extend(preds)
            train_true.extend(labels.cpu().numpy())
            
            loss.backward()
            optimizer.step()
            progress_bar.set_postfix({'loss': loss.item()})
        
        if scheduler:
            scheduler.step()
            
        train_loss /= len(train_dataloader)
        train_preds = np.array(train_preds)
        train_true = np.array(train_true)
        
        train_acc, train_precisions, train_recalls, train_f1s = get_metrics(train_true, train_preds)
        
        start_val = perf_counter()
        model.eval()
        val_loss = 0
        val_preds = []
        val_true = []
        with torch.no_grad():
            for batch in tqdm(val_dataloader, desc="Validation", leave=False):
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                labels = batch['labels'].to(DEVICE)
                
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
                preds = torch.argmax(outputs, dim=1).cpu().numpy()
                val_preds.extend(preds)
                val_true.extend(labels.cpu().numpy())
        
        val_time = perf_counter() - start_val
        
        val_loss /= len(val_dataloader)
        val_preds = np.array(val_preds)
        val_true = np.array(val_true)
        
        val_acc, val_precisions, val_recalls, val_f1s = get_metrics(val_true, val_preds)
        
        print(f"Epoch {epoch + 1}/{epochs} - Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1s}, Prec: {train_precisions}, Recall: {train_recalls}")
        print(f"Epoch {epoch + 1}/{epochs} - Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1s}, Prec: {val_precisions}, Recall: {val_recalls}, Val Time: {val_time:.2f} sec")
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'results/bert_multiclass3.pt')
            print("Model saved!")
    
    total_train_time = perf_counter() - start_train
    print(f"Total Training Time: {total_train_time:.2f} seconds")
    
    return train_acc, train_precisions, train_recalls, train_f1s, val_acc, val_precisions, val_recalls, val_f1s, total_train_time, val_time

In [8]:
def evaluate_model(model, test_dataloader):
    model.eval()
    predictions = []
    true_labels = []
    classification_times = []
    
    start_test = perf_counter()
    
    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc="Testing"):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            
            for i in range(input_ids.size(0)):
                input_id = input_ids[i].unsqueeze(0)
                attention_mask_sample = attention_mask[i].unsqueeze(0)
                label = labels[i].item()
                
                start_time = perf_counter()
                
                output = model(input_ids=input_id, attention_mask=attention_mask_sample)  # Shape: (1, num_classes)
                pred = torch.argmax(output, dim=1).item()  # Scalar integer
                
                predictions.append(pred)
                true_labels.append(label)
                classification_times.append(perf_counter() - start_time)
    
    total_test_time = perf_counter() - start_test
    print(f"Test Time: {total_test_time:.2f} seconds")
    
    predictions = np.array(predictions)
    true_labels = np.array(true_labels)
    
    acc, precisions, recalls, f1s = get_metrics(true_labels, predictions)
    
    print("Test Metrics:")
    print("Accuracy:", acc)
    print("F1s:", f1s)
    print("Precisions:", precisions)
    print("Recalls:", recalls)
    
    return predictions, true_labels

In [9]:
train_texts = train_df['text'].values
train_labels = train_df['label'].values  # Must be integers: 0, 1, 2, ..., num_classes-1

val_texts = val_df['text'].values
val_labels = val_df['label'].values

test_texts = test_df['text'].values
test_labels = test_df['label'].values

tokenizer = DistilBertTokenizer.from_pretrained(PRETRAINED_MODEL_NAME)

# Use MultiClassClassificationDataset instead of BinaryClassificationDataset
train_dataset = MultiClassClassificationDataset(train_texts, train_labels, tokenizer, MAX_LEN)
val_dataset = MultiClassClassificationDataset(val_texts, val_labels, tokenizer, MAX_LEN)
test_dataset = MultiClassClassificationDataset(test_texts, test_labels, tokenizer, MAX_LEN)

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

seeds = [2,3,5]

num_classes = train_df['label'].nunique()

avg_train_acc = 0
avg_train_precs = np.zeros(num_classes)
avg_train_recalls = np.zeros(num_classes)
avg_train_f1s = np.zeros(num_classes)
avg_max_memory_usage_train = 0
avg_max_vram_usage_train = 0
avg_total_train_time = 0

avg_val_acc = 0
avg_val_precs = np.zeros(num_classes)
avg_val_recalls = np.zeros(num_classes)
avg_val_f1s = np.zeros(num_classes)
avg_total_val_time = 0

avg_test_acc = 0
avg_test_precs = np.zeros(num_classes)
avg_test_recalls = np.zeros(num_classes)
avg_test_f1s = np.zeros(num_classes)
avg_max_memory_usage_test = 0
avg_max_vram_usage_test = 0
avg_total_test_time = 0

for seed in seeds:
    torch.manual_seed(seed)
    # Use multi-class model with num_classes parameter
    model = DistilBertForMultiClassClassification(num_classes)
    model = model.to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    # Use CrossEntropyLoss for multi-class
    criterion = nn.CrossEntropyLoss()

    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    max_memory_usage_train, retval = memory_usage(
        (train_model, (model, train_dataloader, val_dataloader, optimizer, criterion), {'epochs': EPOCHS}),
        max_usage=True,
        retval=True
    )

    max_vram_usage_train = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

    (train_acc, train_precisions, train_recalls, train_f1s, 
     val_acc, val_precisions, val_recalls, val_f1s, 
     total_train_time, val_time) = retval

    # Load the best model saved during training (ensure train_model saves to this path)
    model.load_state_dict(torch.load('results/bert_multiclass3.pt'))

    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    start = perf_counter()
    max_memory_usage_test, retval = memory_usage(
        (evaluate_model, (model, test_dataloader), {}),
        max_usage=True,
        retval=True
    )
    total_time_test = perf_counter() - start

    max_vram_usage_test = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

    predictions, true_labels = retval

    test_acc, test_precisions, test_recalls, test_f1s = get_metrics(true_labels, predictions)

    avg_train_acc += train_acc
    avg_train_precs += train_precisions
    avg_train_recalls += train_recalls
    avg_train_f1s += train_f1s
    avg_max_memory_usage_train += max_memory_usage_train
    avg_max_vram_usage_train += max_vram_usage_train
    avg_total_train_time += total_train_time

    avg_val_acc += val_acc
    avg_val_precs += val_precisions
    avg_val_recalls += val_recalls
    avg_val_f1s += val_f1s
    avg_total_val_time += val_time

    avg_test_acc += test_acc
    avg_test_precs += test_precisions
    avg_test_recalls += test_recalls
    avg_test_f1s += test_f1s
    avg_max_memory_usage_test += max_memory_usage_test
    avg_max_vram_usage_test += max_vram_usage_test
    avg_total_test_time += total_time_test

avg_train_acc /= len(seeds)
avg_train_precs /= len(seeds)
avg_train_recalls /= len(seeds)
avg_train_f1s /= len(seeds)
avg_max_memory_usage_train /= len(seeds)
avg_max_vram_usage_train /= len(seeds)
avg_total_train_time /= len(seeds)

avg_val_acc /= len(seeds)
avg_val_precs /= len(seeds)
avg_val_recalls /= len(seeds)
avg_val_f1s /= len(seeds)
avg_total_val_time /= len(seeds)

avg_test_acc /= len(seeds)
avg_test_precs /= len(seeds)
avg_test_recalls /= len(seeds)
avg_test_f1s /= len(seeds)
avg_max_memory_usage_test /= len(seeds)
avg_max_vram_usage_test /= len(seeds)
avg_total_test_time /= len(seeds)

avg_classification_time = avg_total_test_time / len(test_texts)

avg_classification_time

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/3 - Train Loss: 0.4835, Acc: 0.8350, F1: [0.87729876 0.88213065 0.67514248 0.7960199  0.77630511 0.61472785], Prec: [0.82592242 0.85222531 0.78812692 0.85975282 0.81534091 0.7890411 ], Recall: [0.93549078 0.91421112 0.5904908  0.74108384 0.74083634 0.5034965 ]
Epoch 1/3 - Val Loss: 0.1675, Acc: 0.9335, F1: [0.95620438 0.95528744 0.88760807 0.93474427 0.86935867 0.85      ], Prec: [0.95970696 0.95460993 0.9112426  0.90753425 0.87559809 0.86075949], Recall: [0.95272727 0.95596591 0.86516854 0.96363636 0.86320755 0.83950617], Val Time: 7.05 sec
Model saved!


Epoch 2/3 - Train Loss: 0.1466, Acc: 0.9409, F1: [0.97209103 0.95580678 0.86463878 0.9461467  0.90741216 0.81818182], Prec: [0.97376344 0.95768583 0.85746606 0.94834807 0.90163099 0.81818182], Recall: [0.97042435 0.9539351  0.87193252 0.94395553 0.91326794 0.81818182]
Epoch 2/3 - Val Loss: 0.1365, Acc: 0.9345, F1: [0.95934959 0.95609756 0.86322188 0.94332724 0.87793427 0.83333333], Prec: [0.95332136 0.93844049 0.94039735 0.94852941 0.87383178 0.86666667], Recall: [0.96545455 0.97443182 0.79775281 0.93818182 0.88207547 0.80246914], Val Time: 7.10 sec
Model saved!


C:\Users\Rafael\AppData\Local\Temp\ipykernel_16516\556703573.py:73: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('results/bert_multiclass3.

Epoch 3/3 - Train Loss: 0.1081, Acc: 0.9506, F1: [0.98122519 0.96587731 0.88370314 0.95145631 0.91709578 0.81842338], Prec: [0.98238453 0.96569724 0.88167939 0.94970005 0.91474063 0.82944345], Recall: [0.98006858 0.96605744 0.8857362  0.95321908 0.91946309 0.80769231]
Epoch 3/3 - Val Loss: 0.1687, Acc: 0.9335, F1: [0.95357143 0.95731281 0.875      0.92664093 0.89038031 0.84      ], Prec: [0.93684211 0.94344828 0.93037975 0.98765432 0.84680851 0.91304348], Recall: [0.97090909 0.97159091 0.8258427  0.87272727 0.93867925 0.77777778], Val Time: 7.09 sec
Total Training Time: 419.52 seconds


...

Epoch 2/3 - Train Loss: 0.1462, Acc: 0.9381, F1: [0.97163045 0.95459227 0.85498489 0.93781903 0.9024453  0.8246696 ], Prec: [0.97069519 0.95845084 0.8422619  0.93956299 0.89989733 0.8312611 ], Recall: [0.97256751 0.95076464 0.86809816 0.93608152 0.90500774 0.81818182]
Epoch 2/3 - Val Loss: 0.1501, Acc: 0.9365, F1: [0.95777179 0.96218487 0.88685015 0.93357271 0.87022901 0.84615385], Prec: [0.94671403 0.94889503 0.97315436 0.92198582 0.94475138 0.76237624], Recall: [0.96909091 0.97585227 0.81460674 0.94545455 0.80660377 0.95061728], Val Time: 7.05 sec
Model saved!


Epoch 3/3 - Train Loss: 0.1083, Acc: 0.9495, F1: [0.97858672 0.9637093  0.87136294 0.95240306 0.92177481 0.83882458], Prec: [0.97774925 0.96415904 0.87003058 0.95484171 0.91590214 0.85480944], Recall: [0.97942563 0.96325998 0.87269939 0.94997684 0.92772328 0.82342657]
Epoch 3/3 - Val Loss: 0.1474, Acc: 0.9330, F1: [0.96357013 0.95111732 0.84242424 0.93140794 0.89208633 0.86390533], Prec: [0.96532847 0.93543956 0.91447368 0.92473118 0.90731707 0.82954545], Recall: [0.96181818 0.96732955 0.78089888 0.93818182 0.87735849 0.90123457], Val Time: 6.96 sec
Model saved!
Total Training Time: 427.87 seconds


C:\Users\Rafael\AppData\Local\Temp\ipykernel_16516\556703573.py:73: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('results/bert_multiclass3.

Test Time: 12.10 seconds
Test Metrics:
Accuracy: 0.9225
F1s: [0.96302666 0.94416961 0.80272109 0.92028986 0.88940092 0.73239437]
Precisions: [0.96219931 0.92777778 0.87407407 0.91696751 0.91904762 0.68421053]
Recalls: [0.96385542 0.96115108 0.74213836 0.92363636 0.86160714 0.78787879]


0.006277920866666439

In [10]:
# save results to txt
with open("results/bert_multiclass3.txt", "w") as f:
    f.write(f"Average Train Accuracy: {avg_train_acc}\n")
    f.write(f"Average Train Precisions: {avg_train_precs}\n")
    f.write(f"Average Train Recalls: {avg_train_recalls}\n")
    f.write(f"Average Train F1s: {avg_train_f1s}\n")
    f.write(f"Average Max Memory Usage Train: {avg_max_memory_usage_train}\n")
    f.write(f"Average Max VRAM Usage Train: {avg_max_vram_usage_train}\n")
    f.write(f"Average Total Train Time: {avg_total_train_time}\n")
    f.write("\n")
    f.write(f"Average Val Accuracy: {avg_val_acc}\n")
    f.write(f"Average Val Precisions: {avg_val_precs}\n")
    f.write(f"Average Val Recalls: {avg_val_recalls}\n")
    f.write(f"Average Val F1s: {avg_val_f1s}\n")
    f.write(f"Average Total Val Time: {avg_total_val_time}\n")
    f.write("\n")
    f.write(f"Average Test Accuracy: {avg_test_acc}\n")
    f.write(f"Average Test Precisions: {avg_test_precs}\n")
    f.write(f"Average Test Recalls: {avg_test_recalls}\n")
    f.write(f"Average Test F1s: {avg_test_f1s}\n")
    f.write(f"Average Max Memory Usage Test: {avg_max_memory_usage_test}\n")
    f.write(f"Average Max VRAM Usage Test: {avg_max_vram_usage_test}\n")
    f.write(f"Average Total Test Time: {avg_total_test_time}\n")
    f.write("\n")
    f.write(f"Average Classification Time: {avg_classification_time}\n")
    f.write(f"Lines classified {len(test_texts)}\n")

    f.close()